# From a Novel to the Press: Finding Monte-Cristo with Impresso Entities

<a target="_blank" href="https://colab.research.google.com/github/impresso/impresso-datalab-notebooks/blob/main/annotate/NE-processing_ImpressoAPI.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

If something doesn't work, you can [report a problem](https://github.com/impresso/impresso-datalab-notebooks/blob/main/reporting-problems.md).

## What is this notebook about?

Before it was a book, *Le Comte de Monte-Cristo* was a newspaper serial. From August 1844 to January 1846, Alexandre Dumas published it in the *Journal des débats*, one episode at a time, in the *feuilleton* at the bottom of the front page. The *Journal des débats* is part of the Impresso corpus.

In chapter 11, "L'Ogre de Corse", fiction meets history. Paris, March 1815: in his study at the Tuileries, King Louis XVIII is chatting with his favourite, M. de Blacas, when the minister of police bursts in. Napoléon has escaped from Elba and landed at Golfe-Juan.

Starting from this chapter, we ask: **can we follow a page of a novel back into the newspapers: its people, its time, and the page itself?**

We use the Impresso API in two ways:

1. **Annotate our own text** with the Impresso named entity recognition and linking service (NER and NEL). It finds the persons, places and organizations of the text and links them to [Wikidata](https://www.wikidata.org).
2. **Search the corpus** with these Wikidata IDs, and with the meaning of our text. The newspapers of the Impresso corpus were annotated with the same models, so the IDs connect our text to the articles that mention the same people.

The annotation service runs the same models as the [NE-processing_ImpressoHF](https://github.com/impresso/impresso-datalab-notebooks/blob/main/annotate/NE-processing_ImpressoHF.ipynb) notebook, which shows how to run them on your own machine and explains their output in detail.

## What will you learn?

- Annotate a text with the Impresso NER and NEL service, and why a long text must be sent sentence by sentence
- Look up the corpus entities by their Wikidata ID
- Search the articles that mention one or several entities, and count them per year and newspaper
- Normalize counts by the size of the corpus
- Find the articles closest in meaning to your text, with text embeddings
- Compare your annotations with the annotations of the corpus

## Useful resources

- [The Impresso Python Library documentation](https://impresso.github.io/impresso-py/)
- [Introduction to the Impresso Python Library](https://github.com/impresso/impresso-datalab-notebooks/blob/main/starter/basics_ImpressoAPI.ipynb) and [A quick guide to searching with the Impresso library](https://github.com/impresso/impresso-datalab-notebooks/blob/main/starter/search_ImpressoAPI.ipynb)

## Prerequisites

Run the following cells to install the Impresso Python Library and connect to the Impresso API. `connect()` asks for your API token the first time; you can get one from the [Impresso Datalab](https://impresso-project.ch/datalab/).

> If you are working with Google Colab, you may need to restart the session after the installation: go to *Runtime* and select *Restart session*.

In [ ]:
from packaging.version import Version

MIN_VERSION = "0.9.20"
try:
    from impresso import version
    assert Version(version) >= Version(MIN_VERSION)
    print(f"✔ impresso {version} is installed and up to date.")
except (ImportError, AssertionError):
    %pip install --upgrade impresso

In [ ]:
from impresso import connect

client = connect(public_api_url="https://dev.impresso-project.ch/public-api/v1")

In this notebook, we often go back and forth between the notebook and the [Impresso web app](https://impresso-project.ch/app), so a small function that builds the link to an article is useful.

In [ ]:
def article_url(article_id):
    """Link to an article in the Impresso web app."""
    issue_id, item = article_id.rsplit("-", 1)
    return f"https://dev.impresso-project.ch/app/issue/{issue_id}/view?articleId={item}"

# Part 1. Annotating our text

## Our text

Our excerpt is the beginning of chapter 11, "L'Ogre de Corse", as transcribed on [Wikisource](https://fr.wikisource.org/wiki/Le_Comte_de_Monte-Cristo/Chapitre_11) from an 1889 edition of the novel. The text is in the public domain.

In [ ]:
text = """Louis XVIII, à l’aspect de ce visage bouleversé, repoussa violemment la table devant laquelle il se trouvait.
— Qu’avez-vous donc, monsieur le baron ? s’écria-t-il, vous paraissez tout bouleversé : ce trouble, cette hésitation, ont-ils rapport à ce que disait M. de Blacas, et à ce que vient de me confirmer M. de Villefort ?
De son côté M. de Blacas s’approchait vivement du baron, mais la terreur du courtisan empêchait de triompher l’orgueil de l’homme d’État ; en effet, en pareille circonstance, il était bien autrement avantageux pour lui d’être humilié par le préfet de police que de l’humilier sur un pareil sujet.
— Eh bien ! voyons, dit Louis XVIII.
Le ministre de la police, cédant alors à un mouvement de désespoir, alla se précipiter aux pieds de Louis XVIII, qui recula d’un pas en fronçant le sourcil.
— Oh ! sire, quel affreux malheur ! suis-je assez à plaindre ? je ne m’en consolerai jamais !
— Monsieur, dit Louis XVIII, je vous ordonne de parler.
— Eh bien ! sire, l’usurpateur a quitté l’île d’Elbe le 28 février et a débarqué le 1er mars.
— Où cela ? demanda vivement le roi.
— En France, sire, dans un petit port, près d’Antibes, au golfe Juan.
— L’usurpateur a débarqué en France, près d’Antibes, au golfe Juan, à deux cent cinquante lieues de Paris, le 1er mars, et vous apprenez cette nouvelle aujourd’hui seulement 3 mars !… Eh ! Monsieur, ce que vous me dites là est impossible : on vous aura fait un faux rapport, ou vous êtes fou.
— Hélas ! sire, ce n’est que trop vrai !
Louis XVIII fit un geste indicible de colère et d’effroi, et se dressa tout debout, comme si un coup imprévu l’avait frappé en même temps au cœur et au visage.
— En France ! s’écria-t-il, l’usurpateur en France ! Mais on ne veillait donc pas sur cet homme ? mais, qui sait ? on était donc d’accord avec lui !
— Oh ! sire, s’écria le duc de Blacas, ce n’est pas un homme comme M. Dandré que l’on peut accuser de trahison. Sire, nous étions tous aveugles, et le ministre de la police a partagé l’aveuglement général, voilà tout.
— Mais… dit Villefort : puis s’arrêtant tout à coup : Ah ! pardon, pardon, sire, fit-il en s’inclinant, mon zèle m’emporte, que Votre Majesté daigne m’excuser.
— Parlez, Monsieur, parlez hardiment, dit Louis XVIII ; vous seul nous avez prévenu du mal, aidez-nous à y chercher le remède.
— Sire, dit Villefort, l’usurpateur est détesté dans le Midi ; il me semble que s’il se hasarde dans le midi, on peut facilement soulever contre lui la Provence et le Languedoc.
— Oui, sans doute, dit le ministre, mais il s’avance par Gap et Sisteron.
— Il s’avance, il s’avance, dit Louis XVIII ; il marche donc sur Paris ?
Le ministre de la police garda un silence qui équivalait au plus complet aveu.
— Et le Dauphiné, Monsieur, demanda le roi à Villefort, croyez-vous qu’on puisse le soulever comme la Provence ?
— Sire, je suis fâché de dire à Votre Majesté une vérité cruelle ; mais l’esprit du Dauphiné est loin de valoir celui de la Provence et du Languedoc. Les montagnards sont bonapartistes, sire.
— Allons, murmura Louis XVIII, il était bien renseigné. Et combien d’hommes a-t-il avec lui ?
— Sire, je ne sais, dit le ministre de la police.
— Comment, vous ne savez ! Vous avez oublié de vous informer de cette circonstance ? Il est vrai qu’elle est de peu d’importance, ajouta-t-il avec un sourire écrasant.
— Sire, je ne pouvais m’en informer ; la dépêche portait simplement l’annonce du débarquement et de la route prise par l’usurpateur.
— Et comment donc vous est parvenue cette dépêche ? demanda le roi.
Le ministre baissa la tête, et une vive rougeur envahit son front.
— Par le télégraphe, sire, balbutia-t-il.
Louis XVIII fit un pas en avant et croisa les bras comme eût fait Napoléon."""

print(f"{len(text)} characters")

## Annotating with `tools.ner_nel`

`tools.ner_nel` recognizes the entity mentions of a text and links them, in a single call. Let's send it our whole text:

In [ ]:
result = client.tools.ner_nel(text=text)
result.df[["type", "surfaceForm", "wikidata.id"]]

Only a few entities are found, all in the first third of the text: Provence, Dauphiné, Gap, Sisteron and Napoléon, which come later, are missing. The service reads a text as **a single sentence** and stops after 512 sub-tokens, about one or two paragraphs: the rest of the text gets no entities, without any warning.

## Annotating sentence by sentence

The Impresso corpus was annotated sentence by sentence, and we do the same. We split the text into sentences, annotate each one, and shift the offsets so that they point into the full text. A simple regular expression is enough to split the sentences here.

In [ ]:
import re
import pandas as pd


def annotate(text):
    """Annotate `text` sentence by sentence with the Impresso NER and NEL service."""
    rows = []
    for sentence in re.finditer(r"[^.!?]+(?:[.!?]+|$)", text):
        if not sentence.group().strip():
            continue
        for e in client.tools.ner_nel(text=sentence.group()).raw["entities"]:
            rows.append({
                "type": e["type"],
                "surface": e["surfaceForm"],
                "start": sentence.start() + e["offset"]["start"],
                "end": sentence.start() + e["offset"]["end"],
                "confidence_ner": e["confidence"]["ner"],
                "confidence_nel": e["confidence"].get("nel"),
                "wikidata_id": e.get("wikidata", {}).get("id"),
                "wikipedia_page": e.get("wikidata", {}).get("wikipediaPageName"),
            })
    return pd.DataFrame(rows)


entities = annotate(text)
print(f"{len(entities)} entities")
entities

This time, the whole text is annotated. Each row is a mention with its type (`pers`, `loc`, `org`, `prod` or `time`), its surface form and character offsets, two confidence scores (recognition and linking), and its Wikidata ID and Wikipedia page when it is linked.

At the time of writing, three things stand out:

- **History is well linked.** Louis XVIII and Napoléon are linked with full confidence, and the places draw Napoléon's march from the coast to the Alps: golfe Juan, Provence, Dauphiné, Gap, Sisteron.
- **Some real people are missed.** Blacas is recognized as a person, but not linked (`None`), although he has a Wikidata entry. The linker removes a link when it is not sure enough of it.
- **Fiction confuses the linker.** Villefort, the young prosecutor of the novel, is invented. The models don't know him, so they take him for a place and link him to the villages of Villefort in Lozère and in Aude.

The [NE-processing_ImpressoHF](https://github.com/impresso/impresso-datalab-notebooks/blob/main/annotate/NE-processing_ImpressoHF.ipynb) notebook explains these fields, and why a link can be removed.

Here are the Wikidata IDs of our text, one row per entity:

In [ ]:
linked = entities.dropna(subset=["wikidata_id"]).drop_duplicates("wikidata_id")
linked[["surface", "type", "wikidata_id", "wikipedia_page"]]

# Part 2. From our text to the press

## 1. Our persons in the corpus

In the Impresso corpus, an entity is identified by its Wikidata ID, so we can look up the persons of our text directly. `OR(...)` means "any of these IDs".

In [ ]:
from impresso import OR

persons = linked[linked["type"] == "pers"]
client.entities.find(wikidata_id=OR(*persons["wikidata_id"])).df[["label", "type", "totalMentions", "totalContentItems"]]

`totalContentItems` is the number of articles that mention the entity, and `totalMentions` the number of mentions. Blacas, who was not linked in our text, can be looked up by name with `entities.find(term="Blacas")`: at the time of writing, he is not in the corpus.

## 2. The king and the emperor in the press

`search.find(entity_id=...)` returns the articles that mention an entity, and `&` combines two filters, so that the articles must match both. A facet counts the matching articles per value, here per year and per newspaper.

Raw counts per year are misleading, though: the corpus does not have the same number of articles every year. So we also divide them by the number of articles of the whole corpus for the same year.

In [ ]:
wikidata_ids = dict(zip(linked["surface"], linked["wikidata_id"]))
together = OR(wikidata_ids["Louis XVIII"]) & OR(wikidata_ids["Napoléon"])
print("Articles mentioning Louis XVIII and Napoléon:", client.search.find(entity_id=together, limit=1).total)

per_year = client.search.facet("year", entity_id=together, limit=300).df["count"]
all_per_year = client.search.facet("year", limit=500).df["count"]
timeline = pd.DataFrame({
    "articles": per_year,
    "per 10,000 articles": per_year / all_per_year.reindex(per_year.index) * 10_000,
}).sort_index()

timeline.plot(subplots=True, figsize=(10, 5), legend=False,
              title=["Articles mentioning Louis XVIII and Napoléon", "Per 10,000 articles of the corpus"]);

In [ ]:
timeline.sort_values("per 10,000 articles", ascending=False).head(5)

At the time of writing, the top year is **1815**, the year of our scene, in raw counts and even more relative to the size of the corpus, followed by 1814 and 1816: the press of the time followed Napoléon's fall, his return from Elba and the second Restoration of Louis XVIII as current events.

And which newspapers talk about them? `order_by="-count"` puts the largest counts first.

In [ ]:
client.search.facet("newspaper", entity_id=together, order_by="-count", limit=5)

The *Journal des débats*, the newspaper of our serial, is among the first.

## 3. The novel in the press

A novel title is not an entity, so we search for it as a term. Which newspapers mention "Monte-Cristo" while the novel was being serialized?

In [ ]:
from impresso import DateRange

serial = DateRange("1844-08-01", "1846-01-31")
client.search.facet("newspaper", term="Monte-Cristo", date_range=serial, order_by="-count")

In [ ]:
episodes = client.search.find(term="Monte-Cristo", newspaper_id="jdpl", date_range=serial, order_by="date", limit=5)
episodes.df[["meta.date", "text.snippet"]]

Almost all of them are in the *Journal des débats*: they are the episodes themselves. Each one starts with the same header, "Feuilleton du Journal des Débats … LE COMTE DE MONTE-CRISTO", followed by the dates of the previous episodes.

A term search depends on the exact spelling, and on the OCR (the automatic transcription of the printed page). Notice the misread headers above, such as "Feuilleton du.lonrnal des Débats". In some episodes, the title itself is transcribed "MONTE-CHRISTO", and a search for "Monte-Cristo" misses them.

## 4. Finding our page in the newspaper

Entities tell us **who** an article mentions, not **what it is about**. For that, we use text embeddings: a model turns a text into a vector, and texts with similar meanings get similar vectors. The Impresso API gives the embedding of every article, and `tools.embed_text` computes the embedding of our own text with the same model.

`search.find(embedding=...)` returns the articles of the whole corpus whose embeddings are closest to ours. No term, no entity, no date: only the meaning of our text.

In [ ]:
our_embedding = client.tools.embed_text(text=text, target="text")

neighbours = client.search.find(embedding=our_embedding, limit=10)
neighbours.df[["meta.date", "meta.mediaTitle", "relevanceScore", "text.snippet"]]

In [ ]:
original_id = neighbours.df.index[0]
print(original_id, article_url(original_id))

At the time of writing, the closest article of the whole corpus is from the *Journal des débats* of **7 September 1844**: the episode in which our chapter was first printed. Its title is transcribed "MONTE-CHRISTO", so the term search of the previous step did not find it. The second closest is another piece of the same episode, and two more are earlier episodes of the serial (1 and 6 September 1844), with the same characters.

`relevanceScore` measures how close an article is to our text, from 0 to 1 (the same meaning).

## 5. Putting the episode back together

The article we found is only a piece of the episode: the OCR split the long *feuilleton* into several items. To gather the whole episode, we run the same search again, restricted to that issue: same newspaper, same day.

In [ ]:
issue_id = original_id.rsplit("-", 1)[0]          # e.g. jdpl-1844-09-07-a
newspaper, year, month, day = issue_id.split("-")[:4]
issue_day = DateRange(f"{year}-{month}-{day}", f"{year}-{month}-{day}")

issue = client.search.find(embedding=our_embedding, newspaper_id=newspaper, date_range=issue_day, limit=10).df
issue[["text.title", "relevanceScore"]]

Four items stand out, above the rest of the issue (the news, the advertisements, the court reports). They are the four pieces of the episode. We keep them, put them back in their order, and fetch their full text.

In [ ]:
episode_items = issue[issue["relevanceScore"] > 0.87].sort_index()
episode = "\n".join(client.content_items.get(i).raw["text"]["content"] for i in episode_items.index)

print(list(episode_items.index), "\n")
print(episode[:700], "…\n")
signature = episode.index("ALEXANDRE DUMAS")
print("…", episode[signature - 150:signature + 40])

Here is our page as the readers of 1844 discovered it, under the main news, OCR errors included. The episode ends with "ALEXANDRE DUMAS. (La suite à mercredi.)": the next episode came out on Wednesday.

Compare the first lines with our text: in 1844, the king finds the minister "tout ému"; in the 1889 edition, "tout bouleversé". The book is not exactly the serial.

## 6. Our entities and the corpus entities

We now have the same page twice: our text, annotated by us, and the 1844 episode, annotated in the Impresso corpus with the same models. The corpus entities of each item come with the search results, in the `semanticEnrichments.namedEntities` columns.

There is one difference: when the entities are indexed for the Impresso corpus, only the ones the models are sure about are kept: both `confidence_ner` and `confidence_nel` must be above **0.99**. Let's apply this rule to the entities of our text, and compare them with the corpus entities of the episode.

In [ ]:
entities["kept"] = (entities["confidence_ner"] > 0.99) & (entities["confidence_nel"] > 0.99)

corpus_mentions = {}
for column in episode_items.columns:
    if column.startswith("semanticEnrichments.namedEntities."):
        for item_entities in episode_items[column].dropna():
            for e in item_entities:
                corpus_mentions[e["id"]] = corpus_mentions.get(e["id"], 0) + e["count"]

comparison = (entities.dropna(subset=["wikidata_id"])
              .groupby(["wikidata_id", "wikipedia_page"])
              .agg(our_text=("kept", "size"), our_text_above_099=("kept", "sum"))
              .reset_index())
comparison["corpus_episode"] = comparison["wikidata_id"].map(corpus_mentions).fillna(0).astype(int)
comparison.sort_values("our_text", ascending=False)

The entities of our text that pass the 0.99 rule are, almost all, also in the corpus entities of the 1844 episode: Louis XVIII, Napoléon, France, Paris, Provence, Languedoc, golfe Juan. The links below the bar are not in the corpus: the two villages of Villefort, and Sisteron, a correct link that was not certain enough. Blacas, who was not linked in our text, is not in the corpus either.

Two places, the Dauphiné and Gap, pass the rule in our text but are missing from the corpus. The corpus annotated a slightly different text, the 1844 print with its OCR errors, so the scores are not exactly the same. The counts differ too, because the episode goes on after the end of our excerpt.

The corpus favors precision over completeness: what it links is very likely right, but some mentions are not linked. When you annotate your own texts, you can keep all the entities, or filter them on `confidence_ner` and `confidence_nel`, depending on whether you need completeness or certainty.

# Conclusion

Starting from a page of a novel, we went back into the newspapers:

- **Annotation**: the Impresso NER and NEL service turns a text into Wikidata IDs. Send long texts sentence by sentence.
- **Linking**: the Wikidata IDs are the entity IDs of the corpus, and lead to the articles that mention the same persons.
- **Counting**: facets show when and where persons are talked about. Louis XVIII and Napoléon are most often mentioned together in 1815, the year of our scene.
- **Meaning**: embeddings find articles by what they are about. With no other filter, they found the very episode in which our chapter was first printed in 1844, which a term search missed because of the OCR. Restricted to one issue, they put the episode back together.
- **Criticism**: fictional characters lead to wrong links (Villefort becomes a village), and real people can be missed (Blacas). The 0.99 thresholds of the corpus remove the doubtful links, and some correct ones with them: the corpus annotations are a precise but incomplete subset.

> **Takeaway**: Wikidata IDs connect your own text to the Impresso corpus: they tell you who an article is about, and embeddings tell you what it is about.

**Where to go next?**

- Run the models on your own machine, and see their full output: [NE-processing_ImpressoHF](https://github.com/impresso/impresso-datalab-notebooks/blob/main/annotate/NE-processing_ImpressoHF.ipynb)
- Draw the network of persons mentioned together: [Exploring Entity Co-occurrence Networks](https://github.com/impresso/impresso-datalab-notebooks/blob/main/explore-vis/entity_network.ipynb)
- Put the places of your text on a map: [Visualising Place Entities on Maps](https://github.com/impresso/impresso-datalab-notebooks/blob/main/explore-vis/place-entities_map.ipynb)
- Learn more about embeddings: [Understanding Embeddings](https://github.com/impresso/impresso-datalab-notebooks/blob/main/workshop_resources/ws4-embeddings/explore_introduction-to-embeddings.ipynb)

---
## Project and License info

### Notebook credits [CreditLogo.png](https://credit.niso.org/)

**Writing - Original draft:** _to complete_.

<br><a target="_blank" href="https://creativecommons.org/licenses/by/4.0/">
  <img src="https://mirrors.creativecommons.org/presskit/buttons/88x31/png/by.png"  width="100" alt="Open In Colab"/>
</a>

This notebook is published under [CC BY 4.0 License](https://creativecommons.org/licenses/by/4.0/)

For feedback on this notebook, please send an email to info@impresso-project.ch

### Impresso project

[Impresso - Media Monitoring of the Past](https://impresso-project.ch) is an interdisciplinary research project that aims to develop and consolidate tools for processing and exploring large collections of media archives across modalities, time, languages and national borders. The first project (2017-2021) was funded by the Swiss National Science Foundation under grant No. [CRSII5_173719](http://p3.snf.ch/project-173719) and the second project (2023-2027) by the SNSF under grant No. [CRSII5_213585](https://data.snf.ch/grants/grant/213585) and the Luxembourg National Research Fund under grant No. 17498891.
<br></br>
### License

All Impresso code is published open source under the [GNU Affero General Public License](https://github.com/impresso/impresso-pyindexation/blob/master/LICENSE) v3 or later.


---

<p align="center">
  <img src="https://github.com/impresso/impresso.github.io/blob/master/assets/images/3x1--Yellow-Impresso-Black-on-White--transparent.png?raw=true" width="350" alt="Impresso Project Logo"/>
</p>
